In [ ]:
from glob import glob
from os.path import join, basename

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPImageProcessor
from tqdm import tqdm

DATA_DIR = "/kaggle/input/lost-in-the-museum-v2/archive/kaggle_dataset/kaggle_dataset"
CKPT = "openai/clip-vit-base-patch32"
BS = 128
DEV = "cuda" if torch.cuda.is_available() else "cpu"

paths = sorted(glob(join(DATA_DIR, "*.png")))
assert len(paths) == 20000, len(paths)


In [ ]:
class ImgDS(Dataset):
    def __init__(self, paths, proc):
        self.paths = paths
        self.proc = proc

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224))  # битый файл -> серый кадр
        px = self.proc(images=img, return_tensors="pt")["pixel_values"][0]
        return px, i


proc = CLIPImageProcessor.from_pretrained(CKPT)
model = CLIPModel.from_pretrained(CKPT).to(DEV).eval()
dim = model.config.projection_dim


In [ ]:
dl = DataLoader(ImgDS(paths, proc), batch_size=BS, shuffle=False, num_workers=2, pin_memory=DEV == "cuda")
embs = torch.zeros(len(paths), dim)

with torch.no_grad():
    for px, idx in tqdm(dl):
        px = px.to(DEV)
        f = model.get_image_features(pixel_values=px)
        f = f + model.get_image_features(pixel_values=px.flip(-1))  # сумма ~ среднему после L2-нормализации
        embs[idx] = F.normalize(f, dim=-1).cpu()


In [ ]:
names = [basename(p) for p in paths]
cols = [f"feature_{i}" for i in range(dim)]

df = pd.DataFrame(embs.numpy(), columns=cols)
df.insert(0, "image_name", names)
df.insert(0, "ID", names)

assert len(df) == 20000 and df["image_name"].is_unique

df.to_csv("/kaggle/working/submission.csv", index=False, float_format="%.6f")
